In [5]:
from sklearn.metrics import precision_score, accuracy_score,recall_score
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier # using classifier not for regretion purpose

In [7]:
heart= pd.read_csv("heart.csv")

X= heart.drop("target" ,axis=1)
y= heart["target"]

X_train ,X_test, y_train, y_test= train_test_split(
    X,y, random_state=42, test_size=0.2
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [9]:
knn_c = KNeighborsClassifier(n_neighbors=3) # n-neighbors means k
knn_c.fit(X_train_s, y_train)
y_pred= knn_c.predict(X_test_s)

print(f"Precision:{precision_score(y_test,y_pred)}\nAccuracy:{accuracy_score(y_test,y_pred)}\nRecall:{recall_score(y_test,y_pred)}")

# here its not good recall so we change the hyperparameter

Precision:0.9259259259259259
Accuracy:0.8524590163934426
Recall:0.78125


In [11]:
knn_c = KNeighborsClassifier(n_neighbors=5)
knn_c.fit(X_train_s, y_train)
y_pred= knn_c.predict(X_test_s)

print(f"Precision:{precision_score(y_test,y_pred)}\nAccuracy:{accuracy_score(y_test,y_pred)}\nRecall:{recall_score(y_test,y_pred)}")

# know its much better

Precision:0.9333333333333333
Accuracy:0.9016393442622951
Recall:0.875


In [13]:
knn_c = KNeighborsClassifier(n_neighbors=8)
knn_c.fit(X_train_s, y_train)
y_pred= knn_c.predict(X_test_s)

print(f"Precision:{precision_score(y_test,y_pred)}\nAccuracy:{accuracy_score(y_test,y_pred)}\nRecall:{recall_score(y_test,y_pred)}")

Precision:0.9285714285714286
Accuracy:0.8688524590163934
Recall:0.8125


In [15]:
knn_c = KNeighborsClassifier(n_neighbors=7)
knn_c.fit(X_train_s, y_train)
y_pred= knn_c.predict(X_test_s)

print(f"Precision:{precision_score(y_test,y_pred)}\nAccuracy:{accuracy_score(y_test,y_pred)}\nRecall:{recall_score(y_test,y_pred)}")

# best k

Precision:0.9354838709677419
Accuracy:0.9180327868852459
Recall:0.90625


In [23]:
# using cross validation
from sklearn.model_selection import GridSearchCV

clasiffier = KNeighborsClassifier()
param_grid = {"n_neighbors": [3,5,7,9]}

clasiffierCV= GridSearchCV(
    clasiffier, # its the estimator
    param_grid,
    cv= 5 # its the default value (5) which will be used in k fold cv
)
clasiffierCV.fit(X_train_s,y_train)
y_pred= knn_c.predict(X_test_s)

print(f"Precision:{precision_score(y_test,y_pred)}\nAccuracy:{accuracy_score(y_test,y_pred)}\nRecall:{recall_score(y_test,y_pred)}")
res= pd.DataFrame(clasiffierCV.cv_results_)
print(res)
# know we can see its giving the best result without manual try ,in rank at last we can see our cv has selected 5, also we can see that this and manual
# work both are giving diff. ans because cv is first done on train data(80%) 5 times and then on train data, so its much more accurate ,manual is 
# on only the 20% of the data, but if change the scoring to recall it will give 7 again ,but with large dataset this phenomena will do effect.

Precision:0.9354838709677419
Accuracy:0.9180327868852459
Recall:0.90625
   mean_fit_time  std_fit_time  mean_score_time  std_score_time  \
0       0.002035      0.002509         0.007074        0.004890   
1       0.000404      0.000808         0.005525        0.006026   
2       0.000000      0.000000         0.009512        0.007767   
3       0.000000      0.000000         0.008036        0.007576   

  param_n_neighbors              params  split0_test_score  split1_test_score  \
0                 3  {'n_neighbors': 3}           0.816327           0.795918   
1                 5  {'n_neighbors': 5}           0.775510           0.836735   
2                 7  {'n_neighbors': 7}           0.755102           0.857143   
3                 9  {'n_neighbors': 9}           0.734694           0.836735   

   split2_test_score  split3_test_score  split4_test_score  mean_test_score  \
0           0.854167           0.750000           0.812500         0.805782   
1           0.833333        

In [33]:
# more clarity 
print(res[["param_n_neighbors","mean_test_score"]])
print(clasiffierCV.best_params_)

  param_n_neighbors  mean_test_score
0                 3         0.805782
1                 5         0.814116
2                 7         0.801616
3                 9         0.801786
{'n_neighbors': 5}


In [43]:
# using pipeline
from sklearn.pipeline import Pipeline
X_train ,X_test, y_train, y_test= train_test_split(
    X,y, random_state=42, test_size=0.2
)

pipeline = Pipeline([
    ('scaler',StandardScaler()), # this will handle the scalling work
    ('knn',KNeighborsClassifier())
])  # a pipeline takes steps in form of tupple in which order we want it to happen also with names we use to indicate them with

param_grid = {"knn__n_neighbors": [3,5,7,9]}
# here knn with "__" works as a hirarchial separator and tells to use the n_nei. on knn

clasiffierCV= GridSearchCV(
    pipeline, # changed the estimator to pipeline
    param_grid,
    cv= 5 ,
    scoring= "recall"
)
clasiffierCV.fit(X_train,y_train)
y_pred= knn_c.predict(X_test_s)

print(f"Precision:{precision_score(y_test,y_pred)}\nAccuracy:{accuracy_score(y_test,y_pred)}\nRecall:{recall_score(y_test,y_pred)}")
res= pd.DataFrame(clasiffierCV.cv_results_)
print(res)
print(clasiffierCV.best_params_)
# see the same results with no scalling and extra work

Precision:0.9354838709677419
Accuracy:0.9180327868852459
Recall:0.90625
   mean_fit_time  std_fit_time  mean_score_time  std_score_time  \
0       0.006928      0.006724         0.005063        0.003706   
1       0.003159      0.006318         0.009361        0.007644   
2       0.003173      0.006345         0.009559        0.007806   
3       0.005770      0.007126         0.005198        0.005828   

  param_knn__n_neighbors                   params  split0_test_score  \
0                      3  {'knn__n_neighbors': 3}           0.851852   
1                      5  {'knn__n_neighbors': 5}           0.777778   
2                      7  {'knn__n_neighbors': 7}           0.851852   
3                      9  {'knn__n_neighbors': 9}           0.851852   

   split1_test_score  split2_test_score  split3_test_score  split4_test_score  \
0           0.777778           0.925926           0.884615           0.807692   
1           0.814815           0.925926           0.884615           

In [76]:
# implimentation of nieve byes ,logistic regretion ,knn using pipeline
from sklearn.linear_model import LogisticRegression
iris = pd.read_csv("IRIS.csv")

In [84]:
from sklearn.naive_bayes import GaussianNB
X = iris.drop("species",axis=1)
y = iris["species"]

X_train , _, y_train, _= train_test_split(
    X,y, random_state=42, test_size=0.5 # here its listed as half but since we have used "_", we indirectly ignore it
)
X_test, y_test = X,y # this is done to let the model be tested on 100% of the data

#pipelines
pipelines= {
    "knn": Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=5))
    ]),
    "Logistic_reg": Pipeline([
        ('scaler', StandardScaler()),
        ('logreg', LogisticRegression())
    ]),
    "Naive Bayes": Pipeline([
        ('nb', GaussianNB())
    ])
}

# train & evaluate
for name, pipe in pipelines.items(): # for each name (string label like knn) go inside pipe(Pipeline) object and perform the listed tasks
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    print(f"{name} Accuracy: {accuracy_score(y_test, y_pred)}")

knn Accuracy: 0.9466666666666667
Logistic_reg Accuracy: 0.96
Naive Bayes Accuracy: 0.9533333333333334
